# Graph Transformer on Amazon Computers Dataset

Node classification using Graph Transformer (TransformerConv) with multiple head variants.

**Dataset:** Amazon Computers (13,752 nodes, 491,722 edges, 767 features, 10 classes)

## Graph Transformer Baseline (No MLP Head)

In [1]:
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score
from torch_geometric.datasets import Amazon
import torch_geometric.transforms as T
from torch_geometric.nn import TransformerConv, BatchNorm

# 1. Load Amazon Computers Dataset
transform = T.RandomNodeSplit(num_val=0.1, num_test=0.2)
dataset = Amazon(root='./data/Amazon', name='Computers', transform=transform)
data = dataset[0]

# 2. Optimized Architecture (No MLP Head)
class GraphTransformer(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4):
        super().__init__()
        self.conv1 = TransformerConv(in_channels, hidden_channels, heads=heads, dropout=0.5)
        self.bn1 = BatchNorm(hidden_channels * heads)
        self.conv2 = TransformerConv(hidden_channels * heads, out_channels, heads=heads, concat=False, dropout=0.5)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

# 3. Initialize
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GraphTransformer(
    in_channels=dataset.num_node_features,
    hidden_channels=32,
    out_channels=dataset.num_classes,
    heads=4
).to(device)

data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)

# 4. Training with Label Smoothing (Full-Batch)
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask], label_smoothing=0.1)
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    accs = []
    pred_cpu = pred.cpu().numpy()
    y_cpu = data.y.cpu().numpy()
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        m = mask.cpu().numpy()
        accs.append(f1_score(y_cpu[m], pred_cpu[m], average='macro'))
    return accs

# 5. Execution Loop
print(f"Starting Optimized No-MLP Training on {device}...")
best_val_f1 = 0.0

for epoch in range(1, 201):
    loss = train()
    train_f1, val_f1, test_f1 = test()
    scheduler.step(val_f1)
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'Amazon_graphTransformer_noMLP_best.pt')

    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Val F1: {val_f1:.4f}, Best: {best_val_f1:.4f}')

print(f"\nFinal Best Validation Macro-F1: {best_val_f1:.4f}")

ModuleNotFoundError: No module named 'torch'

## Graph Transformer - WITH MLP head (+ Laplacian Positional Encodings)

In [ ]:
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score
from torch_geometric.datasets import Amazon
import torch_geometric.transforms as T
from torch_geometric.nn import TransformerConv, BatchNorm

# 1. Load Amazon Computers Dataset with Laplacian Positional Encodings
# k=8 adds 8 dimensions of structural info to each node
transform = T.Compose([
    T.AddLaplacianEigenvectorPE(k=8, attr_name='pos_enc'),
    T.RandomNodeSplit(num_val=0.1, num_test=0.2)
])

dataset = Amazon(root='./data/Amazon', name='Computers', transform=transform)
data = dataset[0]

# Concatenate original features (767) with Positional Encodings (8)
# New input dimension = 775
data.x = torch.cat([data.x, data.pos_enc], dim=-1)

# 2. Optimized Architecture: Transformer + Residuals + MLP Head
class AdvancedTransformer(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4):
        super().__init__()
        self.conv1 = TransformerConv(in_channels, hidden_channels, heads=heads, dropout=0.5)
        self.bn1 = BatchNorm(hidden_channels * heads)
        self.res1 = torch.nn.Linear(in_channels, hidden_channels * heads)
        
        self.conv2 = TransformerConv(hidden_channels * heads, hidden_channels, heads=heads, concat=False, dropout=0.5)
        self.bn2 = BatchNorm(hidden_channels)
        self.res2 = torch.nn.Linear(hidden_channels * heads, hidden_channels)

        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(hidden_channels, hidden_channels),
            torch.nn.ReLU(),
            torch.nn.Dropout(p=0.7),
            torch.nn.Linear(hidden_channels, out_channels)
        )

    def forward(self, x, edge_index):
        identity = self.res1(x)
        x = self.conv1(x, edge_index) + identity
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        identity = self.res2(x)
        x = self.conv2(x, edge_index) + identity
        x = self.bn2(x)
        x = F.relu(x)
        x = self.mlp(x)
        return x

# 3. Initialization
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AdvancedTransformer(
    in_channels=data.x.size(-1),  # 775 (767 + 8 PE)
    hidden_channels=48,
    out_channels=dataset.num_classes,
    heads=4
).to(device)

data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=7)

# 4. Training (Full-Batch)
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask], label_smoothing=0.1)
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    accs = []
    pred_cpu = pred.cpu().numpy()
    y_cpu = data.y.cpu().numpy()
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        m = mask.cpu().numpy()
        accs.append(f1_score(y_cpu[m], pred_cpu[m], average='macro'))
    return accs

# 5. Execution Loop
print(f"Starting Training: Graph Transformer + PE + Residuals on {device}...")
best_val_f1 = 0.0

for epoch in range(1, 201):
    loss = train()
    train_f1, val_f1, test_f1 = test()
    scheduler.step(val_f1)
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'Amazon_graphTransformer_PE_best.pt')

    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Val F1: {val_f1:.4f}, Best: {best_val_f1:.4f}')

print(f"\nOptimization Complete! Best Val F1 achieved: {best_val_f1:.4f}")

### Graph Transform WITH Residual Head

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score
from torch_geometric.datasets import Amazon
import torch_geometric.transforms as T
from torch_geometric.nn import TransformerConv, BatchNorm

# ──────────────────────────────────────────────────────────────────────
# 1. Load Amazon Computers Dataset
# ──────────────────────────────────────────────────────────────────────
transform = T.RandomNodeSplit(num_val=0.1, num_test=0.2)
dataset = Amazon(root='./data/Amazon', name='Computers', transform=transform)
data = dataset[0]

# ──────────────────────────────────────────────────────────────────────
# 2. Residual Dense Head
# ──────────────────────────────────────────────────────────────────────
class ResidualDenseHead(nn.Module):
    """
    A 3-block dense head with residual skip connections.
    '||' denotes concatenation.
    """
    def __init__(self, in_dim, dense_dim, out_dim, drop_p=0.5):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, dense_dim)
        self.bn1 = nn.BatchNorm1d(dense_dim)
        self.fc2 = nn.Linear(in_dim + dense_dim, dense_dim)
        self.bn2 = nn.BatchNorm1d(dense_dim)
        self.fc3 = nn.Linear(in_dim + dense_dim * 2, dense_dim)
        self.bn3 = nn.BatchNorm1d(dense_dim)
        self.out_proj = nn.Linear(in_dim + dense_dim * 3, out_dim)
        self.drop_p = drop_p

    def forward(self, h):
        d1 = F.relu(self.bn1(self.fc1(h)))
        d1 = F.dropout(d1, p=self.drop_p, training=self.training)
        cat1 = torch.cat([h, d1], dim=-1)
        d2 = F.relu(self.bn2(self.fc2(cat1)))
        d2 = F.dropout(d2, p=self.drop_p, training=self.training)
        cat2 = torch.cat([h, d1, d2], dim=-1)
        d3 = F.relu(self.bn3(self.fc3(cat2)))
        d3 = F.dropout(d3, p=self.drop_p, training=self.training)
        cat3 = torch.cat([h, d1, d2, d3], dim=-1)
        return self.out_proj(cat3)

# ──────────────────────────────────────────────────────────────────────
# 3. GraphTransformer Encoder + Residual Dense Head
# ──────────────────────────────────────────────────────────────────────
class GraphTransformerWithDenseHead(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels,
                 heads=4, dense_dim=64, head_drop=0.5):
        super().__init__()
        self.conv1 = TransformerConv(in_channels, hidden_channels, heads=heads, dropout=0.5)
        self.bn1 = BatchNorm(hidden_channels * heads)
        self.conv2 = TransformerConv(hidden_channels * heads, hidden_channels, heads=heads, concat=False, dropout=0.5)
        self.bn2 = BatchNorm(hidden_channels)
        self.res1 = nn.Linear(in_channels, hidden_channels * heads)
        self.res2 = nn.Linear(hidden_channels * heads, hidden_channels)
        self.head = ResidualDenseHead(in_dim=hidden_channels, dense_dim=dense_dim, out_dim=out_channels, drop_p=head_drop)

    def forward(self, x, edge_index):
        identity = self.res1(x)
        x = self.conv1(x, edge_index) + identity
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        identity = self.res2(x)
        x = self.conv2(x, edge_index) + identity
        x = self.bn2(x)
        x = F.relu(x)
        return self.head(x)

# ──────────────────────────────────────────────────────────────────────
# 4. Initialization
# ──────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GraphTransformerWithDenseHead(
    in_channels=dataset.num_node_features,   # 767
    hidden_channels=32,
    out_channels=dataset.num_classes,        # 10
    heads=4, dense_dim=64, head_drop=0.5,
).to(device)
data = data.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)

# ──────────────────────────────────────────────────────────────────────
# 5. Training (Full-Batch)
# ──────────────────────────────────────────────────────────────────────
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask], label_smoothing=0.1)
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    accs = []
    pred_cpu = pred.cpu().numpy()
    y_cpu = data.y.cpu().numpy()
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        m = mask.cpu().numpy()
        accs.append(f1_score(y_cpu[m], pred_cpu[m], average='macro'))
    return accs

# ──────────────────────────────────────────────────────────────────────
# 6. Execution Loop
# ──────────────────────────────────────────────────────────────────────
print(f"Starting GraphTransformer + Residual Dense Head Training on {device}...")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
best_val_f1 = 0.0
best_test_f1 = 0.0

for epoch in range(1, 201):
    loss = train()
    train_f1, val_f1, test_f1 = test()
    scheduler.step(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_test_f1 = test_f1
        torch.save(model.state_dict(), 'Amazon_graphTransformer_denseHead_best.pt')

    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, '
              f'Train: {train_f1:.4f}, Val: {val_f1:.4f}, '
              f'Test: {test_f1:.4f}, Best Val: {best_val_f1:.4f}')

print(f"\nFinal Best Validation Macro-F1: {best_val_f1:.4f}")
print(f"Test F1uracy at Best Val:      {best_test_f1:.4f}")

### Graph Transform WITH Gated Head

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score
from torch_geometric.datasets import Amazon
import torch_geometric.transforms as T
from torch_geometric.nn import TransformerConv, BatchNorm

# ──────────────────────────────────────────────────────────────────────
# 1. Load Amazon Computers Dataset
# ──────────────────────────────────────────────────────────────────────
transform = T.RandomNodeSplit(num_val=0.1, num_test=0.2)
dataset = Amazon(root='./data/Amazon', name='Computers', transform=transform)
data = dataset[0]

# ──────────────────────────────────────────────────────────────────────
# 2. Gated Head (Highway Networks style)
# ──────────────────────────────────────────────────────────────────────
class GatedBlock(nn.Module):
    """Single gated layer: learns which features to keep vs. transform."""
    def __init__(self, dim, drop_p=0.5):
        super().__init__()
        self.fc_transform = nn.Linear(dim, dim)
        self.bn = nn.BatchNorm1d(dim)
        self.fc_gate = nn.Linear(dim, dim)
        self.drop_p = drop_p

    def forward(self, x):
        t = F.relu(self.bn(self.fc_transform(x)))
        t = F.dropout(t, p=self.drop_p, training=self.training)
        g = torch.sigmoid(self.fc_gate(x))
        return g * t + (1 - g) * x

class GatedHead(nn.Module):
    def __init__(self, in_dim, gate_dim, out_dim, num_blocks=3, drop_p=0.5):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, gate_dim) if in_dim != gate_dim else nn.Identity()
        self.blocks = nn.ModuleList([GatedBlock(gate_dim, drop_p=drop_p) for _ in range(num_blocks)])
        self.out_proj = nn.Linear(gate_dim, out_dim)

    def forward(self, h):
        x = self.input_proj(h)
        for block in self.blocks:
            x = block(x)
        return self.out_proj(x)

# ──────────────────────────────────────────────────────────────────────
# 3. GraphTransformer Encoder + Gated Head
# ──────────────────────────────────────────────────────────────────────
class GraphTransformerWithGatedHead(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels,
                 heads=4, gate_dim=128, num_gate_blocks=3, head_drop=0.5):
        super().__init__()
        self.conv1 = TransformerConv(in_channels, hidden_channels, heads=heads, dropout=0.5)
        self.bn1 = BatchNorm(hidden_channels * heads)
        self.conv2 = TransformerConv(hidden_channels * heads, hidden_channels, heads=heads, concat=False, dropout=0.5)
        self.bn2 = BatchNorm(hidden_channels)
        self.res1 = nn.Linear(in_channels, hidden_channels * heads)
        self.res2 = nn.Linear(hidden_channels * heads, hidden_channels)
        self.head = GatedHead(in_dim=hidden_channels, gate_dim=gate_dim, out_dim=out_channels, num_blocks=num_gate_blocks, drop_p=head_drop)

    def forward(self, x, edge_index):
        identity = self.res1(x)
        x = self.conv1(x, edge_index) + identity
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        identity = self.res2(x)
        x = self.conv2(x, edge_index) + identity
        x = self.bn2(x)
        x = F.relu(x)
        return self.head(x)

# ──────────────────────────────────────────────────────────────────────
# 4. Initialization
# ──────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GraphTransformerWithGatedHead(
    in_channels=dataset.num_node_features,   # 767
    hidden_channels=32,
    out_channels=dataset.num_classes,        # 10
    heads=4, gate_dim=128, num_gate_blocks=3, head_drop=0.5,
).to(device)
data = data.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)

# ──────────────────────────────────────────────────────────────────────
# 5. Training (Full-Batch)
# ──────────────────────────────────────────────────────────────────────
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask], label_smoothing=0.1)
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    accs = []
    pred_cpu = pred.cpu().numpy()
    y_cpu = data.y.cpu().numpy()
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        m = mask.cpu().numpy()
        accs.append(f1_score(y_cpu[m], pred_cpu[m], average='macro'))
    return accs

# ──────────────────────────────────────────────────────────────────────
# 6. Execution Loop
# ──────────────────────────────────────────────────────────────────────
print(f"Starting GraphTransformer + Gated Head Training on {device}...")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
best_val_f1 = 0.0
best_test_f1 = 0.0

for epoch in range(1, 201):
    loss = train()
    train_f1, val_f1, test_f1 = test()
    scheduler.step(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_test_f1 = test_f1
        torch.save(model.state_dict(), 'Amazon_graphTransformer_gatedHead_best.pt')

    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, '
              f'Train: {train_f1:.4f}, Val: {val_f1:.4f}, '
              f'Test: {test_f1:.4f}, Best Val: {best_val_f1:.4f}')

print(f"\nFinal Best Validation Macro-F1: {best_val_f1:.4f}")
print(f"Test F1uracy at Best Val:      {best_test_f1:.4f}")

## MORE optimizations - 3 layer graph transform

In [ ]:
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score
from torch_geometric.datasets import Amazon
import torch_geometric.transforms as T
from torch_geometric.nn import TransformerConv, LayerNorm

# 1. Dataset with higher-dimensional PE
transform = T.Compose([
    T.AddLaplacianEigenvectorPE(k=16, attr_name='pos_enc'),
    T.RandomNodeSplit(num_val=0.1, num_test=0.2)
])

dataset = Amazon(root='./data/Amazon', name='Computers', transform=transform)
data = dataset[0]
data.x = torch.cat([data.x, data.pos_enc], dim=-1)

# 2. Deep & Wide Architecture
class DeepTransformer(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=8):
        super().__init__()
        self.conv1 = TransformerConv(in_channels, hidden_channels, heads=heads, dropout=0.4)
        self.ln1 = LayerNorm(hidden_channels * heads)
        self.res1 = torch.nn.Linear(in_channels, hidden_channels * heads)
        
        self.conv2 = TransformerConv(hidden_channels * heads, hidden_channels, heads=heads, dropout=0.4)
        self.ln2 = LayerNorm(hidden_channels * heads)
        self.res2 = torch.nn.Linear(hidden_channels * heads, hidden_channels * heads)

        self.conv3 = TransformerConv(hidden_channels * heads, hidden_channels, heads=heads, concat=False, dropout=0.4)
        self.ln3 = LayerNorm(hidden_channels)
        self.res3 = torch.nn.Linear(hidden_channels * heads, hidden_channels)

        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(hidden_channels, hidden_channels * 2),
            torch.nn.ReLU(),
            torch.nn.Dropout(p=0.5),
            torch.nn.Linear(hidden_channels * 2, out_channels)
        )

    def forward(self, x, edge_index):
        x = F.relu(self.ln1(self.conv1(x, edge_index) + self.res1(x)))
        x = F.relu(self.ln2(self.conv2(x, edge_index) + self.res2(x)))
        x = F.relu(self.ln3(self.conv3(x, edge_index) + self.res3(x)))
        return self.mlp(x)

# 3. Initialization
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DeepTransformer(
    in_channels=data.x.size(-1),  # 783 (767 + 16 PE)
    hidden_channels=32,
    out_channels=dataset.num_classes,
    heads=8
).to(device)

data = data.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

# 4. Training (Full-Batch)
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask], label_smoothing=0.1)
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    pred_cpu = pred.cpu().numpy()
    y_cpu = data.y.cpu().numpy()
    accs = [f1_score(y_cpu[mask.cpu().numpy()], pred_cpu[mask.cpu().numpy()], average='macro')
            for mask in [data.train_mask, data.val_mask, data.test_mask]]
    return accs

# 5. Execution Loop
best_val_f1 = 0
for epoch in range(1, 201):
    loss = train()
    train_f1, val_f1, test_f1 = test()
    scheduler.step()
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'Amazon_transformer_ultra_best.pt')

    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Val: {val_f1:.4f}, Best: {best_val_f1:.4f}')